In [1]:
# Input Raw Data On Order Report
import pandas as pd
OnOrderReport_df = pd.read_excel("Weekly PTA CGL RTR Revised Open Orders Report 30Aug2026.xlsx", skiprows = 3) 
print(OnOrderReport_df.columns)

Index(['Customer Name', 'Customer Abbreviation', 'Customer Po',
       'Customer Po Line', 'Date Entered', 'Order No', 'Due Date Code',
       'Part No', 'Description', 'Revised Due Date', 'ESD', 'Qty Open',
       'Unit Price', 'Total Value Open', 'Qty Resvered', 'Qty Picked',
       'Qty Backordered', 'Order date', 'Current Sales Status Code',
       'Fin Business Code', 'Carrier Code'],
      dtype='object')


In [2]:
# Make new Sheet PTC Normal
# --- Filter by Customer Name ---
PTCNormal_df = OnOrderReport_df[
    OnOrderReport_df["Customer Name"] == "PT. CIPTA ANDALAN TEKNINDO"
]

In [3]:
# List of new columns to add
new_columns = [
    "Full/Partial",
    "Mon Reqrd.",
    "Mon.Reqrd.2 weeks ago",
    "chk",
    "Year Reqrd.",
    "Mon ListPO",
    "Year 2 weeks ago",
    "chk. Year"
]

# Find position of 'Qty Backordered'
insert_pos = PTCNormal_df.columns.get_loc("Qty Backordered") + 1

# Insert columns one by one (keeps order)
for i, col in enumerate(new_columns):
    PTCNormal_df.insert(insert_pos + i, col, "")
    


In [4]:
import pandas as pd
import numpy as np

# ======================================================
# 1. READ FILES
# ======================================================
al78_df = pd.read_excel("LongPN to AL78 PN.xlsx")

two_weeks_df = pd.read_excel(
    "Open Order Report Edited 16Aug2026 Edited.xlsx",
    sheet_name="PTC Normal")

# IMPORTANT — ensure base df is real copy (fix SettingWithCopyWarning)
PTCNormal_df = PTCNormal_df.copy()

# ======================================================
# 2. CREATE PN used & Order.PN (if not already exist)
# ======================================================
if "PN used" not in PTCNormal_df.columns:
    insert_pos = PTCNormal_df.columns.get_loc("Part No") + 1

    PTCNormal_df.insert(
        insert_pos,
        "PN used",
        PTCNormal_df["Part No"].astype(str).str.strip()
    )

    PTCNormal_df.insert(
        insert_pos + 1,
        "Order.PN",
        PTCNormal_df["Order No"].astype(str).str.strip()
        + "."
        + PTCNormal_df["Part No"].astype(str).str.strip()
    )

# ======================================================
# 3. CLEAN JOIN KEYS
# ======================================================
def clean_col(df, col):
    df.loc[:, col] = df[col].astype(str).str.strip()

for col in ["Order.PN", "PN used"]:
    clean_col(PTCNormal_df, col)

for col in ["Order.PN", "PN used", "PN AL78"]:
    clean_col(two_weeks_df, col)

for col in ["PART_NO", "PN AL78"]:
    clean_col(al78_df, col)

# ======================================================
# 4. BUILD LOOKUPS (STRICT PRIORITY)
# ======================================================

# Priority 1 — Order.PN (2 weeks ago)
lookup_orderpn = (
    two_weeks_df[["Order.PN", "PN AL78"]]
    .dropna()
    .drop_duplicates(subset="Order.PN", keep="last")
)

# Priority 2 — PN used (2 weeks ago) #sort PN used, Ord Date dan Ord Number
lookup_pnused = (
    two_weeks_df[["PN used", "PN AL78"]]
    .dropna()
    .drop_duplicates(subset="PN used", keep="last")
    .rename(columns={"PN AL78": "PN AL78_p2"})
)

# Priority 3 — Master AL78
lookup_master = (
    al78_df[["PART_NO", "PN AL78"]]
    .dropna()
    .drop_duplicates(subset="PART_NO", keep="last")
    .rename(columns={"PN AL78": "PN AL78_p3"})
)

# ======================================================
# 5. APPLY MERGES
# ======================================================

# Merge Priority 1
PTCNormal_df = PTCNormal_df.merge(
    lookup_orderpn,
    on="Order.PN",
    how="left"
)

# Merge Priority 2
PTCNormal_df = PTCNormal_df.merge(
    lookup_pnused,
    on="PN used",
    how="left"
)

# Merge Priority 3
PTCNormal_df = PTCNormal_df.merge(
    lookup_master,
    left_on="PN used",
    right_on="PART_NO",
    how="left"
)

# ======================================================
# 6. FINAL PN AL78 (STRICT PRIORITY ORDER)
# ======================================================
PTCNormal_df["PN AL78"] = (
    PTCNormal_df["PN AL78"]                # Priority 1
    .combine_first(PTCNormal_df["PN AL78_p2"])  # Priority 2
    .combine_first(PTCNormal_df["PN AL78_p3"])  # Priority 3
)

# ======================================================
# 7. CLEANUP TEMP COLUMNS
# ======================================================
PTCNormal_df.drop(
    columns=[
        col for col in [
            "PN AL78_p2",
            "PN AL78_p3",
            "PART_NO"
        ] if col in PTCNormal_df.columns
    ],
    inplace=True
)

# ======================================================
# 8. REORDER — PUT PN AL78 AFTER Order.PN
# ======================================================
if "PN AL78" in PTCNormal_df.columns:
    cols = list(PTCNormal_df.columns)
    cols.remove("PN AL78")

    order_pn_index = cols.index("Order.PN")
    cols.insert(order_pn_index + 1, "PN AL78")

    PTCNormal_df = PTCNormal_df[cols]


In [5]:
#Full/Partial
import numpy as np

PTCNormal_df["Full/Partial"] = np.select(
    [
        PTCNormal_df["Qty Resvered"] == 0,
        PTCNormal_df["Qty Resvered"] == PTCNormal_df["Qty Open"]
    ],
    [
        "00",
        "Full"
    ],
    default="Partial"
)


In [6]:
# --------------------------------------------------
# CLEAN KEYS
# --------------------------------------------------
PTCNormal_df["Order.PN"] = PTCNormal_df["Order.PN"].astype(str).str.strip()
two_weeks_df["Order.PN"] = two_weeks_df["Order.PN"].astype(str).str.strip()

# --------------------------------------------------
# BUILD LOOKUP
# --------------------------------------------------
two_weeks_lookup = (
    two_weeks_df[["Order.PN", "Mon Reqrd."]]
    .rename(columns={"Mon Reqrd.": "Mon.Reqrd.2 weeks ago"})
    .drop_duplicates(subset=["Order.PN"], keep="last")
)

# --------------------------------------------------
# REMOVE OLD COLUMN IF EXISTS
# --------------------------------------------------
if "Mon.Reqrd.2 weeks ago" in PTCNormal_df.columns:
    PTCNormal_df.drop(columns=["Mon.Reqrd.2 weeks ago"], inplace=True)

# --------------------------------------------------
# MERGE
# --------------------------------------------------
PTCNormal_df = PTCNormal_df.merge(
    two_weeks_lookup,
    on="Order.PN",
    how="left",
    validate="many_to_one"
)

# --------------------------------------------------
# MOVE COLUMN AFTER "Mon Reqrd."
# --------------------------------------------------
if {"Mon Reqrd.", "Mon.Reqrd.2 weeks ago"}.issubset(PTCNormal_df.columns):
    cols = list(PTCNormal_df.columns)
    cols.remove("Mon.Reqrd.2 weeks ago")

    idx = cols.index("Mon Reqrd.")
    cols.insert(idx + 1, "Mon.Reqrd.2 weeks ago")

    PTCNormal_df = PTCNormal_df[cols]


In [7]:
# --- Clean join keys ---
PTCNormal_df["Order.PN"] = (
    PTCNormal_df["Order.PN"]
    .astype(str)
    .str.strip()
)

two_weeks_df["Order.PN"] = (
    two_weeks_df["Order.PN"]
    .astype(str)
    .str.strip()
)

# --------------------------------------------------
# BUILD LOOKUP (RENAME BEFORE MERGE 🔑)
# --------------------------------------------------
year_lookup = (
    two_weeks_df[["Order.PN", "Year Reqrd."]]
    .rename(columns={"Year Reqrd.": "Year 2 weeks ago"})
    .dropna(subset=["Order.PN"])
    .drop_duplicates(subset=["Order.PN"], keep="last")
)

# --------------------------------------------------
# REMOVE OLD COLUMN IF EXISTS
# --------------------------------------------------
if "Year 2 weeks ago" in PTCNormal_df.columns:
    PTCNormal_df.drop(columns=["Year 2 weeks ago"], inplace=True)

# --------------------------------------------------
# MERGE (NO _x / _y POSSIBLE)
# --------------------------------------------------
PTCNormal_df = PTCNormal_df.merge(
    year_lookup,
    on="Order.PN",
    how="left",
    validate="many_to_one"
)


In [8]:
import pandas as pd
import numpy as np
from pandas.tseries.offsets import DateOffset

# ======================================================
# 0. READ PO MASTER & BUILD Mon ListPO (VLOOKUP LOGIC)
# ======================================================
po_df = pd.read_excel("PO PTC to 30Aug2026.xlsx", sheet_name="data")

# Clean join keys
PTCNormal_df["Customer Po"] = (
    PTCNormal_df["Customer Po"]
    .astype(str)
    .str.strip()
)

po_df["PO No"] = (
    po_df["PO No"]
    .astype(str)
    .str.strip()
)

# Build lookup table (Excel VLOOKUP behavior)
po_lookup = (
    po_df[["PO No", "MON"]]
    .dropna(subset=["PO No"])
    .drop_duplicates(subset=["PO No"], keep="last")
)

# Remove existing Mon ListPO if exists
if "Mon ListPO" in PTCNormal_df.columns:
    PTCNormal_df.drop(columns=["Mon ListPO"], inplace=True)

# Merge
PTCNormal_df = PTCNormal_df.merge(
    po_lookup,
    left_on="Customer Po",
    right_on="PO No",
    how="left",
    validate="many_to_one"
)

# Rename & cleanup
PTCNormal_df.rename(columns={"MON": "Mon ListPO"}, inplace=True)
PTCNormal_df.drop(columns=["PO No"], inplace=True)


# ======================================================
# 1. ENSURE ORDER DATE
# ======================================================
PTCNormal_df["Order date"] = pd.to_datetime(
    PTCNormal_df["Order date"], errors="coerce"
)
order_year = PTCNormal_df["Order date"].dt.year


# ======================================================
# 2. DUE DATE CODE → DAYS
# ======================================================
due_date_days = {
    "CD": 1, "EC": 3, "WA": 1,
    "MT": 35, "W5": 35,
    "S1": 70, "S2": 70, "S3": 70, "S4": 70, "S5": 70,
    "MO": 35, "TU": 35
}

days_to_add = PTCNormal_df["Due Date Code"].map(due_date_days)


# ======================================================
# 3. FORMULA DATE (LOWEST PRIORITY)
# ======================================================
formula_date = (
    PTCNormal_df["Order date"]
    + pd.to_timedelta(days_to_add, unit="D")
)

formula_month = formula_date.dt.month
formula_year  = formula_date.dt.year


# ======================================================
# 4. CLEAN OVERRIDE INPUTS
# ======================================================
def clean_month(series):
    return (
        pd.to_numeric(series, errors="coerce")
        .where(lambda x: (x >= 1) & (x <= 12))
    )

mon_2w = clean_month(PTCNormal_df["Mon.Reqrd.2 weeks ago"])
mon_listpo = clean_month(PTCNormal_df["Mon ListPO"])

year_2w = pd.to_numeric(
    PTCNormal_df.get("Year 2 weeks ago", pd.Series()),
    errors="coerce"
)


# ======================================================
# 5. FINAL MONTH (PRIORITY LOGIC)
# ======================================================
# Priority: 2-weeks-ago → ListPO → Formula
final_mon = (
    mon_2w
    .combine_first(mon_listpo)
    .combine_first(formula_month)
)


# ======================================================
# 6. FINAL YEAR (PRIORITY LOGIC) — FIXED
# ======================================================
final_year = formula_year.copy()

# Case: Month comes from Mon ListPO (but NOT from 2-weeks-ago)
mask_listpo = mon_2w.isna() & mon_listpo.notna()

# Start from Order year
final_year.loc[mask_listpo] = order_year[mask_listpo]

# 👉 CRITICAL FIX:
# If Mon ListPO month is BEFORE Order date month → roll to next year
roll_mask = (
    mask_listpo
    & (mon_listpo < PTCNormal_df["Order date"].dt.month)
)

final_year.loc[roll_mask] = final_year.loc[roll_mask] + 1

# Year 2 weeks ago ALWAYS wins
final_year.loc[year_2w.notna()] = year_2w[year_2w.notna()]



# ======================================================
# 7. BUILD FINAL DATE
# ======================================================
req_date = pd.to_datetime(
    dict(year=final_year, month=final_mon, day=1),
    errors="coerce"
)


# ======================================================
# 8. APPLY 7-MONTH BACKWARD CAP
# ======================================================
min_allowed_date = (
    pd.Timestamp.today().normalize()
    - DateOffset(months=7)
)

req_date = req_date.where(
    req_date >= min_allowed_date,
    min_allowed_date
)


# ======================================================
# 9. ASSIGN OUTPUT COLUMNS
# ======================================================
PTCNormal_df["Mon Reqrd."] = req_date.dt.month
PTCNormal_df["Year Reqrd."] = req_date.dt.year


# ======================================================
# 10. CLEAN DTYPES (NO .0)
# ======================================================
PTCNormal_df["Mon Reqrd."] = (
    pd.to_numeric(PTCNormal_df["Mon Reqrd."], errors="coerce")
    .astype("Int64")
    .astype(object)
)

PTCNormal_df["Year Reqrd."] = (
    pd.to_numeric(PTCNormal_df["Year Reqrd."], errors="coerce")
    .astype("Int64")
    .astype(object)
)


# ======================================================
# 11. MOVE Mon ListPO NEXT TO Year Reqrd.
# ======================================================
cols = list(PTCNormal_df.columns)
cols.remove("Mon ListPO")

year_reqrd_index = cols.index("Year Reqrd.")
cols.insert(year_reqrd_index + 1, "Mon ListPO")

PTCNormal_df = PTCNormal_df[cols]

In [9]:
PTCNormal_df["chk"] = np.where(
    (PTCNormal_df["Mon Reqrd."].notna()) &
    (PTCNormal_df["Mon Reqrd."] == PTCNormal_df["Mon.Reqrd.2 weeks ago"]),
    "OK",
    "NG"
)

In [10]:
import numpy as np

PTCNormal_df["chk. Year"] = np.where(
    (PTCNormal_df["Year Reqrd."].isna()) &
    (PTCNormal_df["Year 2 weeks ago"].isna()),
    "",
    np.where(
        PTCNormal_df["Year Reqrd."] == PTCNormal_df["Year 2 weeks ago"],
        "OK",
        "NG"
    )
)

In [11]:
# Create Order date 1 and Order date 2 from the same source
PTCNormal_df["Order date 1"] = PTCNormal_df["Order date"]
PTCNormal_df["Order date 2"] = PTCNormal_df["Order date"]

# Optional: drop the original column
PTCNormal_df.drop(columns=["Order date"], inplace=True)
final_columns = [
    "Customer Name",
    "Customer Abbreviation",
    "Customer Po",
    "Customer Po Line",
    "Order date 1",
    "Order No",
    "Due Date Code",
    "Part No",
    "PN used",
    "Order.PN",
    "PN AL78",
    "Description",
    "Revised Due Date",
    "ESD",
    "Qty Open",
    "Unit Price",
    "Total Value Open",
    "Qty Resvered",
    "Qty Picked",
    "Qty Backordered",
    "Full/Partial",
    "Mon Reqrd.",
    "Mon.Reqrd.2 weeks ago",
    "chk",
    "Year Reqrd.",
    "Mon ListPO",
    "Year 2 weeks ago",
    "chk. Year",
    "Order date 2",
    "Date Entered",
    "Current Sales Status Code",
    "Fin Business Code",
    "Carrier Code"
]

PTCNormal_df = PTCNormal_df[[c for c in final_columns if c in PTCNormal_df.columns]]

def clean_int_column(df, col):
    df[col] = (
        pd.to_numeric(df[col], errors="coerce")
        .astype("Int64")   # pandas nullable integer
    )

# Apply to affected columns
clean_int_column(PTCNormal_df, "Mon.Reqrd.2 weeks ago")
clean_int_column(PTCNormal_df, "Mon ListPO")
clean_int_column(PTCNormal_df, "Year 2 weeks ago")

In [12]:
print(PTCNormal_df.columns)

Index(['Customer Name', 'Customer Abbreviation', 'Customer Po',
       'Customer Po Line', 'Order date 1', 'Order No', 'Due Date Code',
       'Part No', 'PN used', 'Order.PN', 'PN AL78', 'Description',
       'Revised Due Date', 'ESD', 'Qty Open', 'Unit Price', 'Total Value Open',
       'Qty Resvered', 'Qty Picked', 'Qty Backordered', 'Full/Partial',
       'Mon Reqrd.', 'Mon.Reqrd.2 weeks ago', 'chk', 'Year Reqrd.',
       'Mon ListPO', 'Year 2 weeks ago', 'chk. Year', 'Order date 2',
       'Date Entered', 'Current Sales Status Code', 'Fin Business Code',
       'Carrier Code'],
      dtype='object')


In [13]:
from datetime import datetime

today_str = datetime.today().strftime("%Y-%m-%d")
output_file = f"PTC Normal Draft.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    PTCNormal_df.to_excel(
        writer,
        sheet_name="PTC Normal",
        index=False
    )

print(f"File saved as: {output_file}")

File saved as: PTC Normal Draft.xlsx
